# E-Commerce Customer & Sales Analysis

## Overview
This analysis explores customer behaviour, revenue trends, and product performance for an e-commerce business using SQL (via pandasql) and Python.

The goal is to surface actionable insights across four key areas:
- **Customer value** — who is spending the most and where are they coming from?
- **Revenue trends** — how is the business growing month on month?
- **Product performance** — which products are rated highest and returned most?
- **Channel effectiveness** — which acquisition channels produce the most valuable customers?

**Dataset:** E-Commerce Customer Behavior and Sales 2020–2026  
**Tables used:** `customers`, `orders`, `product_summary`, `monthly_revenue`

## Setup
Load libraries and the four datasets used throughout this analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

orders       = pd.read_csv('/kaggle/input/datasets/meruvakodandasuraj/e-commerce-customer-behavior-and-sales-20202026/orders.csv')
product      = pd.read_csv('/kaggle/input/datasets/meruvakodandasuraj/e-commerce-customer-behavior-and-sales-20202026/product_summary.csv')
customers    = pd.read_csv('/kaggle/input/datasets/meruvakodandasuraj/e-commerce-customer-behavior-and-sales-20202026/customers.csv')
monthly_rev  = pd.read_csv('/kaggle/input/datasets/meruvakodandasuraj/e-commerce-customer-behavior-and-sales-20202026/monthly_revenue.csv')

## Data Exploration
Before writing any analytical queries, we inspect the structure of each table — column names, data types, and a basic statistical summary. This step ensures we understand what we're working with before drawing conclusions.

In [ ]:
customers.info()
print(customers.describe())


In [ ]:
monthly_rev.info()
print(monthly_rev.describe())


In [ ]:
orders.info()
print(orders.describe())


In [ ]:
product.info()
print(product.describe())


## 1. Revenue by Country

**Business question:** Which markets generate the most revenue?

Understanding geographic revenue distribution helps prioritise where to focus marketing spend, localisation efforts, and customer support resources.

In [ ]:
query = """
SELECT country, SUM(total_spend_usd) AS total_spend
FROM customers
GROUP BY country
ORDER BY total_spend DESC
"""

country_spend = pysqldf(query)
country_spend

In [ ]:
# Bar chart
country_spend.plot(x='country', y='total_spend', kind='bar', figsize=(10,5), legend=False)
plt.title('Total Spend by Country')
plt.ylabel('Total Spend (USD)')
plt.xlabel('Country')
plt.tight_layout()
plt.show()

# Pie chart
plt.pie(country_spend['total_spend'], labels=country_spend['country'], autopct='%1.1f%%')
plt.title('Spend by Country')
plt.show()

Drilling further — spend by country broken down by preferred product category reveals whether category preferences differ across markets.

In [ ]:
query = """
SELECT country, preferred_category, SUM(total_spend_usd) AS total_spend
FROM customers
GROUP BY country, preferred_category
ORDER BY total_spend DESC
"""

df_plot = pysqldf(query)
df_pivot = df_plot.pivot(index='country', columns='preferred_category', values='total_spend').fillna(0)
df_pivot.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Spend by Country and Category')
plt.ylabel('Total Spend (USD)')
plt.tight_layout()
plt.show()

## 2. Spend by Acquisition Channel

**Business question:** Which channels are bringing in the highest value customers?

Acquisition channel analysis helps the marketing team understand which channels deserve more investment and which may be attracting low-value customers despite high traffic volumes.

In [ ]:
query = """
SELECT acquisition_channel, SUM(total_spend_usd) AS total_spend
FROM customers
GROUP BY acquisition_channel
ORDER BY total_spend DESC
"""

pysqldf(query).plot(x='acquisition_channel', y='total_spend', kind='bar', legend=False)
plt.title('Total Spend by Acquisition Channel')
plt.ylabel('Total Spend (USD)')
plt.tight_layout()
plt.show()

## 3. Spend by Membership Tier and Device

**Business question:** Do customers on different membership tiers behave differently depending on the device they use?

This cross-analysis helps product and UX teams prioritise device optimisation efforts for the highest value customer segments.

In [ ]:
query = """
SELECT membership_tier, preferred_device AS device,
       SUM(total_orders) AS orders,
       SUM(total_spend_usd) AS spend
FROM customers
GROUP BY membership_tier, device
ORDER BY spend DESC
"""

pysqldf(query)

## 4. Customers Without Orders

**Business question:** Are there customers who have registered but never placed an order?

Identifying inactive accounts helps target re-engagement campaigns. A `LEFT JOIN` between customers and orders returns `NULL` order values for any customer with no purchase history.

> **Note:** This dataset contains no unactivated customers — every customer has at least one order. The query logic is validated below, and this finding is noted for completeness.

In [ ]:
query = """
SELECT c.customer_id, c.total_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_date IS NULL
GROUP BY c.customer_id
ORDER BY c.total_orders ASC
"""

# Returns empty — all customers in this dataset have placed at least one order
pysqldf(query)

## 5. Top 20% Customer Spend Share

**Business question:** How much of total revenue comes from the top 20% of customers?

This analysis uses `NTILE(5)` to split customers into five equal spend buckets and calculates what share of total revenue comes from the top bucket.

**Finding:** The top 20% of customers account for **60% of total revenue** — stronger than the classic 80/20 rule. This concentration means retaining high value customers is critical; losing even a small number would have an outsized impact on revenue.

In [ ]:
query = """
WITH customer_spend AS (
    SELECT customer_id, SUM(total_spend_usd) AS total_spend
    FROM customers
    GROUP BY customer_id
),
top_bucket AS (
    SELECT customer_id, total_spend,
           NTILE(5) OVER (ORDER BY total_spend DESC) AS spend_bucket
    FROM customer_spend
)
SELECT
    ROUND(
        SUM(CASE WHEN spend_bucket = 1 THEN total_spend ELSE 0 END)
        / SUM(total_spend) * 100
    , 2) AS top_20_pct_share
FROM top_bucket
"""

pysqldf(query)

Identifying who those top 20% customers are — their membership tier and how long they have been registered:

In [ ]:
query = """
WITH customer_spend AS (
    SELECT customer_id, SUM(total_spend_usd) AS total_spend,
           membership_tier, registration_date
    FROM customers
    GROUP BY customer_id, membership_tier, registration_date
),
buckets AS (
    SELECT customer_id, total_spend, membership_tier, registration_date,
           NTILE(5) OVER (ORDER BY total_spend DESC) AS spend_bucket
    FROM customer_spend
)
SELECT customer_id, total_spend, membership_tier, registration_date
FROM buckets
WHERE spend_bucket = 1
ORDER BY total_spend DESC
"""

pysqldf(query)

## 6. Monthly Revenue Trend & Month-on-Month Growth

**Business question:** How has revenue trended over time and where are the growth inflection points?

Using the `LAG()` window function, each month's revenue is compared to the previous month to calculate growth percentage. The trend line shows modest but consistent revenue growth over the period.

In [ ]:
query = """
WITH monthly AS (
    SELECT year, month, revenue_usd,
           LAG(revenue_usd) OVER (ORDER BY year, month) AS prev_month_revenue
    FROM monthly_rev
)
SELECT year, month, revenue_usd,
       ROUND(((revenue_usd - prev_month_revenue) / prev_month_revenue) * 100, 2) AS growth_pct
FROM monthly
ORDER BY year, month
"""

df = pysqldf(query)
df

In [ ]:
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[2, 1], hspace=0.05)

# Top chart - Revenue line with trend
ax1 = fig.add_subplot(gs[0])
ax1.plot(range(len(df)), df['revenue_usd'], color='darkorange', linewidth=2.5, marker='o', markersize=3)
z = np.polyfit(range(len(df)), df['revenue_usd'], 1)
p = np.poly1d(z)
ax1.plot(range(len(df)), p(range(len(df))), color='darkorange', linewidth=1.5, linestyle='--', alpha=0.6, label='Trend')
ax1.set_ylabel('Revenue (USD)', color='darkorange')
ax1.set_xticks([])
ax1.set_title('Monthly Revenue & Month on Month Growth')
ax1.legend()

# Bottom chart - Growth bars
ax2 = fig.add_subplot(gs[1])
colors = ['red' if x < 0 else 'steelblue' for x in df['growth_pct'].fillna(0)]
ax2.bar(range(len(df)), df['growth_pct'].fillna(0), color=colors, alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_ylabel('Growth %', color='steelblue')
labels = [f"{row.year}" if row.month == 1 else '' for _, row in df.iterrows()]
ax2.set_xticks(range(len(df)))
ax2.set_xticklabels(labels, rotation=0, ha='center', fontsize=9)

plt.show()

## 7. Customer Spend vs Country Average

**Business question:** Which customers are significantly above or below the average spend for their country?

Using `AVG() OVER (PARTITION BY country)`, each customer's spend is compared to the average for their country without collapsing the data into a single row per country. This keeps every customer visible alongside their country benchmark.

Customers significantly above average are candidates for premium upsell. Those below average may benefit from re-engagement offers.

In [ ]:
query = """
SELECT
    customer_id,
    country,
    total_spend_usd,
    ROUND(AVG(total_spend_usd) OVER (PARTITION BY country), 2) AS country_avg_spend,
    ROUND(total_spend_usd - AVG(total_spend_usd) OVER (PARTITION BY country), 2) AS diff_from_avg
FROM customers
ORDER BY country, diff_from_avg DESC
"""

pysqldf(query)

## 8. Top Customers by Membership Tier

**Business question:** Who are the highest value customers in each membership tier?

`RANK() OVER (PARTITION BY membership_tier)` assigns a rank to each customer within their tier independently. Knowing the top spenders within each tier helps account managers prioritise relationships and identify customers ready to upgrade.

In [ ]:
query = """
WITH top_three AS (
    SELECT customer_id, membership_tier, total_spend_usd,
           RANK() OVER (PARTITION BY membership_tier ORDER BY total_spend_usd DESC) AS tier_rank
    FROM customers
)
SELECT customer_id, membership_tier, total_spend_usd, tier_rank
FROM top_three
WHERE tier_rank <= 3
ORDER BY membership_tier, tier_rank
"""

df_ranks = pysqldf(query)
df_ranks

In [ ]:
tiers = df_ranks['membership_tier'].unique()
x = np.arange(3)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, tier in enumerate(tiers):
    tier_data = df_ranks[df_ranks['membership_tier'] == tier].sort_values('tier_rank')
    ax.bar(x + i * width, tier_data['total_spend_usd'], width=width, label=tier, alpha=0.8)

ax.set_xlabel('Rank within Tier')
ax.set_ylabel('Total Spend (USD)')
ax.set_title('Top 3 Customers by Spend within Each Membership Tier')
ax.set_xticks(x + width)
ax.set_xticklabels(['Rank 1', 'Rank 2', 'Rank 3'])
ax.legend(title='Membership Tier')
plt.tight_layout()
plt.show()

## 9. Top 5 Products by Rating within Each Category

**Business question:** Which products are customers most satisfied with in each category?

Highly rated products should be prioritised in marketing materials and featured prominently on category pages. Understanding what makes these products successful can inform future buying decisions.

In [ ]:
query = """
WITH top_five AS (
    SELECT product_name, category, avg_rating,
           RANK() OVER (PARTITION BY category ORDER BY avg_rating DESC) AS rating_rank
    FROM product
)
SELECT product_name, category, avg_rating, rating_rank
FROM top_five
WHERE rating_rank <= 5
ORDER BY category, rating_rank
"""

pysqldf(query)

## 10. Most Returned Products by Category

**Business question:** Which products have the highest return rates within each category?

High return rates are costly — they affect margins, logistics, and customer satisfaction. Products with persistently high return rates should be reviewed for quality issues, misleading descriptions, or sizing problems. Cross-referencing return rates with customer ratings can help diagnose the root cause.

In [ ]:
query = """
WITH top_returns AS (
    SELECT product_name, category, avg_rating, return_rate,
           RANK() OVER (PARTITION BY category ORDER BY return_rate DESC) AS return_rank
    FROM product
)
SELECT product_name, category, avg_rating, return_rank
FROM top_returns
WHERE return_rank <= 3
ORDER BY category, return_rank
"""

pysqldf(query)

## 11. Average Order Value by Acquisition Channel and Membership Tier

**Business question:** Which combination of acquisition channel and membership tier produces the highest average order value?

This query joins `customers` and `orders` on `customer_id` — the first multi-table analysis in this project. An `INNER JOIN` is used because we only want customers who have placed orders. The result directly informs budget allocation across acquisition channels and helps the CRM team target the right membership tier upgrades.

In [ ]:
query = """
SELECT
    c.acquisition_channel,
    c.membership_tier,
    COUNT(o.order_id) AS total_orders,
    ROUND(AVG(o.total_amount_usd), 2) AS avg_order_value
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY membership_tier, acquisition_channel
ORDER BY avg_order_value DESC
"""

pysqldf(query)

## Summary & Key Findings

| Finding | Business Insight |
|---|---|
| Top 20% of customers drive 60% of revenue | Retention of high value customers is critical |
| Revenue shows a consistent upward trend | Business is in a healthy growth phase |
| Acquisition channel affects order value | Marketing budget should follow highest value channels |
| Return rates vary significantly by product | Quality review needed for high return rate items |
| Spend patterns differ across countries | Localised marketing strategies may improve conversion |

**Tools used:** SQL (via pandasql), Python, pandas, matplotlib  
**Techniques:** GROUP BY, JOINs, CTEs, window functions (LAG, RANK, NTILE, PARTITION BY), conditional aggregation (CASE WHEN)